# Decorators

In [1]:
def mi_decorador(funcion_original):
    
    def wrapper():
        print("-> ANTES: Abriendo el regalo...")
        funcion_original()
        print("-> DESPUÉS: ¡Regalo entregado!")
        
    return wrapper

In [2]:
def saludar():
    print("¡Hola! Soy la función original.")

In [3]:
saludar()

¡Hola! Soy la función original.


In [23]:
@mi_decorador
def saludar():
    print("¡Hola! Soy la función original.")

In [24]:
saludar()

-> ANTES: Abriendo el regalo...
¡Hola! Soy la función original.
-> DESPUÉS: ¡Regalo entregado!


# Context managers

In [25]:
class Habitacion:
    
    def __enter__(self):
        print("Abriendo la puerta de la habitación...")
        
    def __exit__(self, tipo_error, valor_error, traza_error):
        print("Cerrando la puerta con llave al salir.")

In [26]:
with Habitacion() as llave:
    print(f"¡Estoy dentro!")

Abriendo la puerta de la habitación...
¡Estoy dentro!
Cerrando la puerta con llave al salir.


# Generators

In [27]:
sales_rows = {
    "venta_1": {"cantidad": 2, "precio": 10.5},
    "venta_2": {"cantidad": 3, "precio": 4.0},
    "venta_3": {"cantidad": 1, "precio": 7.5},
}

In [28]:
total = sum(
    int(row.get("cantidad", 0)) * float(row.get("precio", 0.0))
    for row in sales_rows.values()
)

In [ ]:
a = int(row.get("cantidad", 0)) * float(row.get("precio", 0.0)) for row in sales_rows.values()

# Generar Spotify refresh tocken

In [4]:
CLIENT_ID = "TU_CLIENT_ID"
CLIENT_SECRET = "TU_CLIENT_SECRET"
REDIRECT_URI = "http://127.0.0.1:8765/callback"
SCOPES = "playlist-read-private user-read-recently-played"

In [ ]:
import base64
import json
import secrets
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from queue import Empty, Queue
from threading import Thread
from urllib.error import HTTPError
from urllib.parse import parse_qs, urlencode, urlparse
from urllib.request import Request, urlopen

if CLIENT_ID.startswith("TU_") or CLIENT_SECRET.startswith("TU_"):
    raise ValueError("Rellena CLIENT_ID y CLIENT_SECRET antes de ejecutar la celda.")

state = secrets.token_urlsafe(24)
callback_result = Queue()

class Callback(BaseHTTPRequestHandler):
    def do_GET(self):
        parsed = urlparse(self.path)
        query = parse_qs(parsed.query)

        if parsed.path != "/callback" or query.get("state", [None])[0] != state:
            self.send_error(400, "Respuesta de autorización no válida")
            return

        callback_result.put(query)
        message = "Autorización recibida. Vuelve al notebook."
        body = message.encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "text/plain; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, *_args):
        pass

server = ThreadingHTTPServer(("127.0.0.1", 8765), Callback)
Thread(target=server.serve_forever, daemon=True).start()

try:
    url = "https://accounts.spotify.com/authorize?" + urlencode({
        "response_type": "code",
        "client_id": CLIENT_ID,
        "redirect_uri": REDIRECT_URI,
        "scope": SCOPES,
        "state": state,
    })
    print("Abre esta URL en el navegador y acepta los permisos:\n", url, flush=True)
    query = callback_result.get(timeout=300)
finally:
    server.shutdown()
    server.server_close()

if "error" in query:
    detail = query.get("error_description", ["Sin detalle adicional"])[0]
    request_id = query.get("ubi", ["No disponible"])[0]
    raise RuntimeError(
        f"Spotify devolvió {query['error'][0]} antes de emitir un código. "
        f"Detalle: {detail}. Referencia: {request_id}"
    )

code = query.get("code", [None])[0]
if not code:
    raise RuntimeError("Spotify no devolvió el código de autorización.")

credentials = base64.b64encode(
    f"{CLIENT_ID}:{CLIENT_SECRET}".encode("utf-8")
).decode("ascii")

request = Request(
    "https://accounts.spotify.com/api/token",
    data=urlencode({
        "grant_type": "authorization_code",
        "code": code,
        "redirect_uri": REDIRECT_URI,
    }).encode("utf-8"),
    headers={
        "Authorization": f"Basic {credentials}",
        "Content-Type": "application/x-www-form-urlencoded",
    },
)

try:
    with urlopen(request, timeout=20) as response:
        tokens = json.load(response)
except HTTPError as error:
    details = json.loads(error.read().decode("utf-8"))
    raise RuntimeError(f"Spotify: {details.get('error_description', details.get('error'))}") from None

print("\nSPOTIFY_REFRESH_TOKEN =", tokens["refresh_token"])